In [ ]:
!pip install spacy
!python -m spacy download en_core_web_trf

In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

df = pd.read_parquet("data/news_with_topics_labeled.parquet")

nlp = spacy.load("en_core_web_trf")

In [ ]:
records = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    text = row["article_text"][:3000]
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT"]:
            records.append({
                "article_id": idx,
                "entity": ent.text.strip(),
                "entity_type": ent.label_,
                "topic": row["topic"],
                "topic_label": row["topic_label"]
            })

entities = pd.DataFrame(records)

entities.head()

In [ ]:
entities["entity_clean"] = (
    entities["entity"]
    .str.replace(r"[^A-Za-z0-9&.\- ]", "", regex=True)
    .str.strip()
)

entities = entities[entities["entity_clean"].str.len() >= 2]

In [ ]:
top_entities = (
    entities["entity_clean"]
    .value_counts()
    .reset_index()
)

top_entities.columns = ["entity", "mentions"]

top_entities.head(30)

In [ ]:
entities.to_csv("outputs/entities_all.csv", index=False)
top_entities.to_csv("outputs/top_entities.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

top20 = top_entities.head(20).sort_values("mentions")

plt.figure(figsize=(10,7))
plt.barh(top20["entity"], top20["mentions"])
plt.xlabel("Mentions")
plt.title("Most Mentioned Organizations / Technologies")
plt.tight_layout()
plt.savefig("figures/top_entities.png", dpi=300)
plt.show()